# DAF-17 in pictures

## 📖 The story: Asha madam and the board-exam rule

Asha madam is the school coordinator. Every evening at 6 she decides: will the 600 children play outside tomorrow or not? Her model gives her tomorrow's PM2.5 number, and she turns it into "go out" or "stay in".

Her model has some **settings** that she can change, like the knobs on a pressure cooker. `alpha` for Ridge. `max_depth` and `learning_rate` for the tree model. With the default knobs the model catches 109 of the 120 Poor days. She thinks: *"If I turn the knobs properly, maybe it catches more."*

So she does the obvious thing. She turns a knob, checks the score on the **test days**, turns another knob, checks again. Twenty times. She keeps the best one.

Then Sharma sir, the old maths teacher, hears about it and laughs.

> *"Beta, this is exactly what students do with the board paper. Suppose a student sees the real board paper in advance, and then plans her study around it. She will score 95. Is she a 95-marks student? No. She is a student who saw the paper. Tomorrow, in college, with a new paper, she will get 60.*
>
> *Good students do not look at the board paper. They write **mock tests** during the year. And they write them in the right order: first Chapter 1 to 3, then test on Chapter 4. Then Chapter 1 to 4, test on Chapter 5. They never write the Chapter 4 test after reading Chapter 5. That would be cheating, and the marks would be fake."*

Asha madam understands in one second. Her 130 test days are the **board paper**. She has looked at it twenty times. It can no longer tell her the truth.

## 🎯 The exact problem

> **How do we choose the best settings for the model, without ever looking at the test days, and without the mock tests cheating by mixing future days into the past?**

## 💡 The new concepts, in Sharma sir's words

| In the story | In our project | The name |
|---|---|---|
| The board paper, never seen before the exam day | The 130 test days (from 1 Mar 2026), locked until DAF-18 | **test set** |
| Mock tests during the year | Fit on some train days, score on other train days | **cross-validation** |
| Mock tests in the right order: Ch 1-3 then test Ch 4, Ch 1-4 then test Ch 5 | Each fold trains on earlier days, validates on the days right after | **`TimeSeriesSplit`** |
| Mixing the chapters up, so you have already read Ch 5 when you write the Ch 4 test | Shuffled `KFold`: tomorrow's days sit inside the training set | the cheat we avoid |
| Different study plans (more revision, more practice papers) | Different settings: `alpha`, `max_depth`, `learning_rate` … | **hyper-parameters** |
| Trying 12 study plans, writing a mock test for each | Try 12 random settings, each scored on 5 folds | **`RandomizedSearchCV`** |
| Choosing the final study plan and writing it on paper **before** the exam day | Saving the winner in `models/final_config.json` before DAF-18 | **freeze the config** |

That is the whole ticket. The rest of this notebook is eight pictures, drawn on the real Delhi data (the 251 training days, 17 Mar 2025 to 28 Feb 2026). Read them in order.

| # | The question the picture answers | Story line |
|---|---|---|
| 1 | Which days are the "mock test" days and which is the "board paper"? | the board paper is locked |
| 2 | Why is the shuffled mock test fake? | reading Ch 5 before the Ch 4 test |
| 3 | What do the proper mock tests look like? | the five tests in order |
| 4 | Are the tests cut by calendar or by number of rows? | the number of pages, not the date |
| 5 | What happens inside one mock test? | one test, start to marks |
| 6 | How does one study plan get one score? | five tests, one average |
| 7 | How many mock tests does a search cost? | 12 plans × 5 tests |
| 8 | In what order do I finish? | write the plan, then open the paper |

The pictures use Ridge at three alphas only because it is quick and real. The real tuning is in `17_tuning.ipynb`.

## 1 · Where tuning is allowed

![1 · Where tuning is allowed](figures/17_walkthrough/p01_where_tuning_lives.png)

**Look at:** the cut date, 1 Mar 2026. Everything left of it (251 days) is the only data this ticket may touch. Everything right of it (130 days) stays locked until DAF-18.

| | Rows | Dates | Poor days |
|---|---:|---|---:|
| Train | 251 | 17 Mar 2025 – 28 Feb 2026 | 117 |
| Test (locked) | 130 | 1 Mar 2026 – 25 Aug 2026 | 17 |

**Why it matters:** each time you look at a test score and change a setting, the test days quietly become part of the training signal. After twenty looks, the best setting is partly luck on those exact 130 days.

**In the story:** this is the board paper (right, hatched) and the whole year of study material (left, blue).

## 2 · Why shuffled `KFold` cheats

![2 · Why shuffled KFold cheats](figures/17_walkthrough/p02_why_kfold_cheats.png)

**Look at:** the two blue dots next to the red one. With a shuffled `KFold`, 20 Nov (221) and 22 Nov (270) are in the training set when the model is asked about 21 Nov (238). Smog lasts for days, so the neighbours almost give the answer away.

**In `TimeSeriesSplit`** the grey days after 21 Nov are hidden. That is the situation in real life: Asha has only the past.

**Result of cheating:** the CV score looks better than the model really is, and the tuning picks settings that are good at *interpolating*, not at *forecasting*.

**In the story:** this is the student who has already read Chapter 5 when she writes the Chapter 4 test. Her marks are fake.

## 3 · The five folds

![3 · The five folds](figures/17_walkthrough/p03_the_five_folds.png)

**Look at:** every blue block sits to the right of its grey block. Fold 1 trains on 46 rows and fold 5 on 210. The grey block only ever grows.

| Fold | Train rows | Validation dates (41 rows each) | Poor days |
|---:|---:|---|---:|
| 1 | 46 | 11 Jun – 13 Aug 2025 | 3 |
| 2 | 87 | 14 Aug – 23 Sep 2025 | 0 |
| 3 | 128 | 24 Sep – 6 Nov 2025 | 23 |
| 4 | 169 | 7 Nov – 17 Dec 2025 | 41 |
| 5 | 210 | 18 Dec 2025 – 28 Feb 2026 | 39 |

**Two things to notice.** No fold reaches the dashed line, so the test period is never used. And the folds are not alike: fold 2 has no Poor day, fold 4 has only Poor days.

**In the story:** these are the five mock tests, always written after the chapters they cover. Fold 1 is the first small test, fold 5 the last big one.

## 4 · Rows, not days

![4 · Rows, not days](figures/17_walkthrough/p04_rows_not_days.png)

**Look at:** the dark stripes. 349 calendar days minus 98 missing days is 251 rows. `TimeSeriesSplit` counts **rows**, so each validation block is 41 rows, but it spans 41 to 73 calendar days (fold 1: 64 days, fold 5: 73 days, because of gaps).

**Why it matters:** equal row counts are not equal calendar spans. Do not describe a fold as "two months" without checking its dates.

**In the story:** each test has the same number of questions (41), but the chapters do not take the same number of days to cover.

## 5 · Inside one fold (fold 4)

![5 · Inside one fold](figures/17_walkthrough/p05_inside_one_fold.png)

**Look at:** the four boxes. Cut, fit on 169 rows, predict 41 days, score. The validation rows are not shown to the model until step 3. The model is then thrown away.

| Box | Real numbers |
|---|---|
| Cut | train 17 Mar – 6 Nov 2025 (169 rows), validation 7 Nov – 17 Dec 2025 (41 rows) |
| Predict | 7 Nov: real 264, forecast 181 · 27 Nov: real 240, forecast 235 · 17 Dec: real 345, forecast 215 |
| Score | MAE 52.26 µg/m³ |

The scaler is fitted inside the fold too, on the 169 train rows only. That is why the tuning notebook will use a `Pipeline`.

**In the story:** one mock test from start to marks: study, write, check, and then you close the notebook.

## 6 · One setting, five folds, one score

![6 · One setting, five folds, one score](figures/17_walkthrough/p06_setting_times_folds.png)

**Look at:** the bar heights change a lot between folds (15 to 60) but the **order** of the three colours is almost the same in each fold. Fold difficulty is not the model's fault. Compare the averages, not one fold.

| Ridge alpha | Mean of 5 folds | Best fold | Worst fold |
|---:|---:|---:|---:|
| 0.1 | 36.41 | 14.6 | 52.3 |
| 10 | 36.89 | 15.2 | 53.2 |
| 100 | 39.86 | 17.8 | 59.8 |

These are real, default-style numbers. Alpha 0.1 and 10 are only 0.5 apart: tuning Ridge may help very little. That is a result worth reporting honestly in the notebook.

**In the story:** a study plan is judged by the average over all five mock tests, not by the one test that happened to be easy or hard.

## 7 · What a search costs

![7 · What a search costs](figures/17_walkthrough/p07_randomized_search.png)

**Look at:** the grid. One row is one candidate setting, one column is one fold. Every cell is one fit. 12 candidates × 5 folds = 60 fits, and the last column is the number the search ranks by.

The ranges drawn are **examples**. You choose the real ones for Ridge (`alpha`) and for the tree model, from the table in the ticket.

**In the story:** Asha madam tries 12 study plans. Each plan needs 5 mock tests, so she writes 60 tests in all.

## 8 · Finish in this order

![8 · Finish in this order](figures/17_walkthrough/p08_freeze_then_open.png)

**Look at:** the dashed line before DAF-18. Steps 1 to 3 happen on train rows only. The config file is written *before* the test is opened, so nobody can claim the settings were chosen after seeing the test score.

**The trap:** "just checking" the test score after tuning. Once you have looked, those 130 days stop being a test set.

**In the story:** she writes the final study plan on a paper, signs it, and only then opens the board paper.